In [1]:
import os
import dotenv
import duckdb
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely
import logging
import uuid
import gc 

gc.enable()

gpd.options.io_engine = "pyogrio"
os.environ["PYOGRIO_USE_ARROW"] = "1"

dotenv.load_dotenv()

ret_class=os.getenv("RET_CLASS")
wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
aflb_parquet=os.getenv("ALFB_PARQUET")
cia_dissolve=os.getenv("CIA_DISSOLVE")
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

In [3]:
#variables 
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
input_data_gdb=os.path.join(source_data,'Scenario5_Setup.gdb')
cia_ciap_l='CIA_CIAP_noHV1TU'
hv1_l='HV1_Clip'
tsu_l='trapline_clip'


In [8]:
conn = duckdb.connect()
info('connected')
conn.install_extension("httpfs")
conn.install_extension("spatial")
conn.load_extension("spatial")
conn.load_extension("httpfs")


2026-02-12 08:47:16,575 - INFO - connected


IOException: IO Error: Failed to download extension "httpfs" at URL "http://extensions.duckdb.org/v1.0.0/windows_amd64/httpfs.duckdb_extension.gz"
Extension "httpfs" is an existing extension.
 (ERROR Connection)

In [ ]:
#query for all records
sql = f"""
    SELECT 
        *, ST_AsText(geometry) as wkt,
    FROM '{ret_class}'
    """
    
#query for step 2a
sql_2a=f"""
    SELECT *,ST_AsText(geometry) as wkt, from '{ret_class}'
    WHERE
    Rec_Cat = 1 And Rec_Cat_short IN ('HPC', 'HPD', 'HPM', 'LPC', 'LPD', 'LPM', 'MPC', 'MPD', 'MPM')
    OR Rec_Cat = 2 And Rec_Cat_short IN ('HPC', 'HPD', 'HPM', 'MPC', 'MPD', 'LPD')
    OR Rec_Cat = 3 And Rec_Cat_short IN ('LPD', 'HPC', 'HPD', 'HPM')
    """

#query for step 2b
sql_2b=f"""
    SELECT *,ST_AsText(geometry) as wkt, from '{ret_class}'
    WHERE
    Rec_Cat = 2 And Rec_Cat_short IN ('MPM', 'LPC', 'LPM')
    Or Rec_Cat = 3 And Rec_Cat_short IN ('MPC', 'MPD') 
    Or Rec_Cat = 4 And Rec_Cat_short IN ('HPC', 'HPD', 'HPM', 'LPD')
    """
    
    

In [ ]:
def return_gdf(sql):
    df = conn.sql(sql).to_df()
    df['geometry'] = gpd.GeoSeries.from_wkt(df['wkt'])
    df = gpd.GeoDataFrame(df).set_crs(3005, allow_override=True)
    df = df.drop(columns=['wkt'])
    info(f" length of df: {len(df)}")
    return df
    

In [ ]:

def pick_oldest_closest(grp: pd.DataFrame, area_col) -> pd.DataFrame:
    """
    Sort by SIFA_2 descending (oldest first), pick contiguous prefix with sum(Area_ha)
    closest to target_ha. If exact tie, prefer undershoot.
    """
    grp = grp.sort_values(['SIFA_2', area_col], ascending=[False, False]).copy()
    target = grp['target_ha'].iloc[0]

    if grp.empty:
        grp['cum_area_ha'] = 0.0
        grp['picked'] = False
        grp['picked_area_sum'] = 0.0
        grp['delta_to_target'] = -target
        return grp

    grp['cum_area_ha'] = grp[area_col].cumsum()

    # count of rows with cum_area <= target
    floor_n = (grp['cum_area_ha'] <= target).sum()

    if floor_n == 0:
        # First polygon already overshoots; choose it because it's the closest allowed
        idx_to_pick = [grp.index[0]]
    elif floor_n == len(grp):
        # Even all don't reach target (no overshoot possible); pick all
        idx_to_pick = list(grp.index)
    else:
        # Compare undershoot vs. adding the next oldest (overshoot)
        under_sum = grp['cum_area_ha'].iloc[floor_n - 1]
        over_sum  = grp['cum_area_ha'].iloc[floor_n]
        if (over_sum - target) < (target - under_sum):
            idx_to_pick = list(grp.index[:floor_n + 1])
        elif (over_sum - target) > (target - under_sum):
            idx_to_pick = list(grp.index[:floor_n])
        else:
            # Tie → prefer undershoot
            idx_to_pick = list(grp.index[:floor_n])

    grp['picked'] = False
    grp.loc[idx_to_pick, 'picked'] = True
    grp['picked_area_sum'] = grp.loc[grp['picked'], area_col].sum()
    grp['delta_to_target'] = grp['picked_area_sum'] - target
    return grp



In [ ]:
#step 2a
ret_class_2a=return_gdf(sql_2a)
ret_class_2a.to_parquet(os.path.join(output_dir, "ret_class_2a.parquet"))

In [ ]:
targets=pd.read_csv(brwmb_targets_csv)
targets_long = (
    targets.query("Rec_Cat != 'Total AFLB (ha)'")
    .assign(rec_Cat=lambda d: pd.to_numeric(d['Rec_Cat'], errors='coerce'))
    .melt(id_vars='Rec_Cat', var_name='Rec_Cat_short', value_name='target_ha')
    .dropna(subset=['Rec_Cat', 'target_ha'])
)

targets_long['Rec_Cat'] = targets_long['Rec_Cat'].astype('Int64')
targets_long['Rec_Cat_short'] = targets_long['Rec_Cat_short'].astype(str).str.strip()
targets_long['target_ha'] = pd.to_numeric(targets_long['target_ha'], errors='coerce').fillna(0)


targets_2b = targets_long.query(
    "(Rec_Cat == 2 and Rec_Cat_short in ['MPM','LPC','LPM'])"
    " or (Rec_Cat == 3 and Rec_Cat_short in ['MPC','MPD'])"
    " or (Rec_Cat == 4 and Rec_Cat_short in ['HPC','HPD','HPM','LPD'])"
).copy()

In [ ]:
#step2b
ret_class_2b=return_gdf(sql_2b)
ret_class_2b['SIFA_2'] = pd.to_numeric(ret_class_2b['SIFA_2'], errors='coerce')
ret_class_2b['Area_ha'] = pd.to_numeric(ret_class_2b['Area_ha'], errors='coerce')
ret_class_2b = ret_class_2b.dropna(subset=['SIFA_2', 'Area_ha'])
ret_class_2b['Rec_Cat_short'] = ret_class_2b['Rec_Cat_short'].astype(str).str.strip()
ret_class_2b['Rec_Cat'] = pd.to_numeric(ret_class_2b['Rec_Cat'], errors='coerce').astype('Int64')




gdf2 = ret_class_2b.merge(
    targets_2b[['Rec_Cat','Rec_Cat_short','target_ha']],
    on=['Rec_Cat', 'Rec_Cat_short'],
    how='inner'
)

selected = (
    gdf2
    .groupby(['Rec_Cat', 'Rec_Cat_short', 'target_ha'], group_keys=False, sort=False)
    .apply(lambda grp: pick_oldest_closest(grp, 'Area_ha'))
    .reset_index(drop=True)
    
)

selection_gdf = selected[selected['picked']].copy()

# create summary table with metrics comparing to targets
summary = (
    selection_gdf
    .groupby(['Rec_Cat', 'Rec_Cat_short', 'target_ha'], as_index=False)
    .agg(selected_area_ha=('Area_ha', 'sum'),
         n_features=('Area_ha', 'size'),
         max_SIFA2=('SIFA_2', 'max'),
         min_SIFA2=('SIFA_2', 'min'))
)
summary['delta_to_target'] = summary['selected_area_ha'] - summary['target_ha']
summary['pct_met'] = (summary['selected_area_ha'] / summary['target_ha']) * 100.0

#write selected to geoparquet
selection_gdf.to_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))

#check summary columns
# for s in summary.itertuples():
#     info(f"Summary - Rec_Cat: {s.Rec_Cat}, Rec_Cat_short: {s.Rec_Cat_short}, target_ha: {s.target_ha}, selected_area_ha: {s.selected_area_ha}, n_features: {s.n_features}, max_SIFA2: {s.max_SIFA2}, min_SIFA2: {s.min_SIFA2}, delta_to_target: {s.delta_to_target}")
# # pair = selection_gdf.query("Rec_Cat == 4 and Rec_Cat_short == 'HPC'")
# # print("Rec_Cat=4/HPC — Target:", targets_long.query("Rec_Cat==4 and Rec_Cat_short=='HPC'")['target_ha'].iloc[0])
# # print("Rec_Cat=4/HPC — Picked sum(Area_ha):", pair['Area_ha'].sum())
# # print("Rec_Cat=4/HPC — n features:", len(pair))
# # print("Rec_Cat=4/HPC — SIFA_2 range:", (pair['SIFA_2'].min(), pair['SIFA_2'].max()))

#join summary to targets, then write out summary with metrics as csv
summary_wide = (
    summary
    .assign(
        metric_selected=lambda d: d['Rec_Cat_short'] + '_selected',
        metric_minSIFA=lambda d: d['Rec_Cat_short'] + '_minSIFA2',
        metric_maxSIFA=lambda d: d['Rec_Cat_short'] + '_maxSIFA2',
        metric_delta=lambda d: d['Rec_Cat_short'] + '_delta',
        metric_pctmet=lambda d: d['Rec_Cat_short'] + '_pct_met'
    )
)


# Stack into long-with-metric, then pivot to wide (columns are metrics)
long_metrics = pd.concat([
    summary_wide[['Rec_Cat', 'metric_selected', 'selected_area_ha']].rename(columns={'metric_selected':'metric', 'selected_area_ha':'value'}),
    summary_wide[['Rec_Cat', 'metric_minSIFA', 'min_SIFA2']].rename(columns={'metric_minSIFA':'metric', 'min_SIFA2':'value'}),
    summary_wide[['Rec_Cat', 'metric_maxSIFA', 'max_SIFA2']].rename(columns={'metric_maxSIFA':'metric', 'max_SIFA2':'value'}),
    summary_wide[['Rec_Cat', 'metric_delta', 'delta_to_target']].rename(columns={'metric_delta':'metric', 'delta_to_target':'value'}),
    summary_wide[['Rec_Cat', 'metric_pctmet', 'pct_met']].rename(columns={'metric_pctmet':'metric', 'pct_met':'value'}),
], ignore_index=True)

wide_metrics = (
    long_metrics
    .pivot_table(index='Rec_Cat', columns='metric', values='value', aggfunc='first')
    .reset_index()
)
wide_metrics['Rec_Cat'] = pd.to_numeric(wide_metrics['Rec_Cat'], errors='coerce').astype('Int64')


# Split numeric rows vs total row
is_total = targets['Rec_Cat'].astype(str).str.contains('Total', case=False, na=False)
targets_numeric = targets.loc[~is_total].copy()
targets_total   = targets.loc[ is_total].copy()  # keep as-is

# Normalize types for the merge
targets_numeric['Rec_Cat'] = pd.to_numeric(targets_numeric['Rec_Cat'], errors='coerce').astype('Int64')

# Merge metrics
augmented_numeric = targets_numeric.merge(wide_metrics, on='Rec_Cat', how='left')

# Reassemble full table (numeric rows first, then the Total row)
augmented_full = pd.concat([augmented_numeric, targets_total], ignore_index=True)

# Optional: order columns so that the original class columns stay first
# and the added metrics come after.
orig_cols = ['Rec_Cat'] + [c for c in targets.columns if c != 'Rec_Cat']
metric_cols = [c for c in augmented_full.columns if c not in orig_cols]
augmented_full = augmented_full[orig_cols + sorted(metric_cols)]
augmented_full.to_csv(os.path.join(output_dir, "step_2b_summary.csv"), index=False)

#setp 2c 

In [ ]:
cia_ciap=gpd.read_parquet(cia_dissolve)

In [ ]:
hv1_dv=gpd.read_file(input_data_gdb, where="zone='Development'", layer=hv1_l)

In [ ]:
tsu=gpd.read_file(input_data_gdb, layer=tsu_l)
tsu=tsu.dissolve()

In [ ]:

selected=gpd.read_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))

In [ ]:
#merge cia and tsu
cia_tsu= gpd.overlay(cia_ciap, tsu, how='union')
info(f"After merging CIA_CIAP_noHV1TU and trapline_clip, cia_tsu has {len(cia_tsu)} features")
# select all stands that intersect with cia_tsu
ret_class_2c=gpd.overlay(selected, cia_tsu, how='intersection')
info(f"After intersecting selected with cia_tsu, ret_class_2c has {len(ret_class_2c)} features")

In [ ]:
#remove features that intersect with hv1
ret_class_2c_nohv1=gpd.overlay(ret_class_2c, hv1_dv, how='difference')
info(f"After removing features that intersect with HV1_Clip, ret_class_2c_nohv1 has {len(ret_class_2c_nohv1)} features")

In [ ]:

#remove layers not longer needed to free up memory
# del ret_class_2c
# del cia_ciap
# del cia_tsu
# del hv1_dv
# del tsu
# del ret_class_2a
# del ret_class_2b
# del targets
# del targets_long
# del gdf2
# del selected
# del summary
# del summary_wide
# del long_metrics
# del wide_metrics
gc.collect()

#get ret_class_2c_nohv1 bbox for sql query
minx, miny, maxx, maxy = ret_class_2c_nohv1.total_bounds

minx_s = f"{minx:.6f}"
miny_s = f"{miny:.6f}"
maxx_s = f"{maxx:.6f}"
maxy_s = f"{maxy:.6f}"



sql_aflb_bbox = f"""
WITH bbox AS (
  SELECT ST_MakeEnvelope(
           CAST({minx_s} AS DOUBLE),
           CAST({miny_s} AS DOUBLE),
           CAST({maxx_s} AS DOUBLE),
           CAST({maxy_s} AS DOUBLE)
         ) AS geom
)
SELECT afl.*,
       ST_AsText(afl.geometry) AS wkt
FROM read_parquet('{aflb_parquet}') AS afl, bbox
WHERE ST_Intersects(
  ST_Envelope(afl.geometry),
  bbox.geom
);
"""

aflb_gdf = return_gdf(sql_aflb_bbox)



In [ ]:
aflb_gdf.to_parquet(os.path.join(output_dir, "step_2c_aflb_gdf.parquet"))

In [ ]:
# give each feaure in ret_class_2c_nohv1 a unqie id 
ret_class_2c_nohv1.ret = ret_class_2c_nohv1.to_crs(3005).copy()
ret_class_2c_nohv1["uid"] = np.arange(1, len(ret_class_2c_nohv1) + 1, dtype="int64")
# ret_class_2c_nohv1['uid'] = [uuid.uuid4().hex for _ in range(len(ret_class_2c_nohv1))]
info(f"Assigned unique IDs to ret_class_2c_nohv1, now has {len(ret_class_2c_nohv1)} features with uid")


# intersect ret_class_2c_nohv1 with aflb_gdf only want to join the attribute aflb_fact
# rc_2c_aflb= gpd.overlay(ret_class_2c_nohv1, aflb_gdf, how='union')
# info(f"After intersecting ret_class_2c_nohv1 with aflb_gdf, rc_2c_aflb has {len(rc_2c_aflb)} features")

ret_tbl = pd.DataFrame({
    "uid": ret_class_2c_nohv1["uid"].values,
    "geom_wkb": ret_class_2c_nohv1.geometry.map(shapely.to_wkb)
})
aflb_tbl = pd.DataFrame({
    "aflb_fact": pd.to_numeric(aflb_gdf["aflb_fact"], errors="coerce").fillna(0).values,
    "geom_wkb": aflb_gdf.geometry.map(shapely.to_wkb)
})


conn.register("ret_tbl", ret_tbl)
conn.register("aflb_tbl", aflb_tbl)


sql = """
WITH ret AS (
  SELECT uid, ST_GeomFromWKB(geom_wkb) AS g FROM ret_tbl
),
aflb AS (
  SELECT aflb_fact, ST_GeomFromWKB(geom_wkb) AS g FROM aflb_tbl
),
pairs AS (
  -- bbox accelerated intersects; DuckDB/GEOS will short-circuit many non-overlaps
  SELECT
    r.uid,
    a.aflb_fact,
    ST_Area(ST_Intersection(r.g, a.g)) AS area_m2
  FROM ret r
  JOIN aflb a
    ON ST_Intersects(r.g, a.g)
)
SELECT
  uid,
  SUM(aflb_fact * (area_m2 / 10000.0)) AS aflb_area -- hectares
FROM pairs
GROUP BY uid
ORDER BY uid;
"""

aflb_area_df = conn.execute(sql).df()
ret = ret_class_2c_nohv1.merge(aflb_area_df, on="uid", how="left")
ret["aflb_area"] = ret["aflb_area"].fillna(0)

info(f"Computed AFLB area contribution for {len(ret)} polygons (sum ha: {ret['aflb_area'].sum():,.2f})")


In [ ]:

ret=ret.dropna(subset=['uid'])
info(f"After dropping records without uid, ret has {len(ret)} features")

ret_dissolved = ret.dissolve(by='uid', aggfunc={'aflb_area': 'sum'})
info(f"Dissolved ret by uid, now has {len(ret_dissolved)} features with summed aflb_area by uid")

ret=gpd.sjoin(ret_dissolved, ret_class_2c_nohv1, how='left', predicate='crosses')

cols_to_drop = [
    c for c in ret.columns
    if c.endswith('_2') and c != 'SIFA_2'
]

ret = ret.drop(columns=cols_to_drop)


if 'Ret_Cat' not in ret.columns:
    ret = ret.rename(columns={'Rec_Cat_1': 'Rec_Cat'})

if 'Rec_Cat_short' not in ret.columns:
    ret = ret.rename(columns={'Rec_Cat_short_1': 'Rec_Cat_short'})

In [ ]:
    #inputs gdf to calculate on, targets df (targets_2b), and output csv path for summary table with metrics comparing to targets
ret['SIFA_2'] = pd.to_numeric(ret['SIFA_2'], errors='coerce')
ret['Area_ha'] = pd.to_numeric(ret['aflb_area'], errors='coerce')
ret = ret.dropna(subset=['SIFA_2', 'aflb_area'])
ret['Rec_Cat_short'] = ret['Rec_Cat_short'].astype(str).str.strip()
ret['Rec_Cat'] = pd.to_numeric(ret['Rec_Cat'], errors='coerce').astype('Int64')


    
gdf2 = ret.merge(
    targets_2b[['Rec_Cat','Rec_Cat_short','target_ha']],
    on=['Rec_Cat', 'Rec_Cat_short'],
    how='inner'
)

if 'target_ha' not in gdf2.columns:
    gdf2 = gdf2.rename(columns={'target_ha_y': 'target_ha'})
    
    
selected = (
    gdf2
    .groupby(['Rec_Cat', 'Rec_Cat_short', 'target_ha'], group_keys=False, sort=False)
    .apply(lambda grp: pick_oldest_closest(grp, 'aflb_area'))
    .reset_index(drop=True)
    
)

selection_gdf = selected[selected['picked']].copy()

# create summary table with metrics comparing to targets
summary = (
    selection_gdf
    .groupby(['Rec_Cat', 'Rec_Cat_short', 'target_ha'], as_index=False)
    .agg(selected_aflb_area=('aflb_area', 'sum'),
        n_features=('aflb_area', 'size'),
        max_SIFA2=('SIFA_2', 'max'),
        min_SIFA2=('SIFA_2', 'min'))
)
summary['delta_to_target'] = summary['selected_aflb_area'] - summary['target_ha']
summary['pct_met'] = (summary['selected_aflb_area'] / summary['target_ha']) * 100.0

#write selected to geoparquet
selection_gdf.to_parquet(os.path.join(output_dir, "ret_class_2c.parquet"))

summary_wide = (
    summary
    .assign(
        metric_selected=lambda d: d['Rec_Cat_short'] + '_selected',
        metric_minSIFA=lambda d: d['Rec_Cat_short'] + '_minSIFA2',
        metric_maxSIFA=lambda d: d['Rec_Cat_short'] + '_maxSIFA2',
        metric_delta=lambda d: d['Rec_Cat_short'] + '_delta',
        metric_pctmet=lambda d: d['Rec_Cat_short'] + '_pct_met'
    )
)

long_metrics = pd.concat([
    summary_wide[['Rec_Cat', 'metric_selected', 'selected_aflb_area']].rename(
        columns={'metric_selected':'metric', 'selected_aflb_area':'value'}
    ),
    summary_wide[['Rec_Cat', 'metric_minSIFA', 'min_SIFA2']].rename(
        columns={'metric_minSIFA':'metric', 'min_SIFA2':'value'}
    ),
    summary_wide[['Rec_Cat', 'metric_maxSIFA', 'max_SIFA2']].rename(
        columns={'metric_maxSIFA':'metric', 'max_SIFA2':'value'}
    ),
    summary_wide[['Rec_Cat', 'metric_delta', 'delta_to_target']].rename(
        columns={'metric_delta':'metric', 'delta_to_target':'value'}
    ),
    summary_wide[['Rec_Cat', 'metric_pctmet', 'pct_met']].rename(
        columns={'metric_pctmet':'metric', 'pct_met':'value'}
    ),
], ignore_index=True)


wide_metrics = (
    long_metrics
    .pivot_table(index='Rec_Cat', columns='metric', values='value', aggfunc='first')
    .reset_index()
)
wide_metrics['Rec_Cat'] = pd.to_numeric(wide_metrics['Rec_Cat'], errors='coerce').astype('Int64')

is_total = targets_2b['Rec_Cat'].astype(str).str.contains('Total', case=False, na=False)
targets_numeric = targets_2b.loc[~is_total].copy()
targets_total   = targets_2b.loc[ is_total].copy()  # keep as-is

targets_numeric['Rec_Cat'] = pd.to_numeric(targets_numeric['Rec_Cat'], errors='coerce').astype('Int64')

augmented_numeric = targets_numeric.merge(wide_metrics, on='Rec_Cat', how='left')

augmented_full = pd.concat([augmented_numeric, targets_total], ignore_index=True)

orig_cols = ['Rec_Cat'] + [c for c in targets_2b.columns if c != 'Rec_Cat']
metric_cols = [c for c in augmented_full.columns if c not in orig_cols]
augmented_full = augmented_full[orig_cols + sorted(metric_cols)]
augmented_full.to_csv(os.path.join(output_dir, "step_2c_summary.csv"), index=False)

In [ ]:
# Short Fall Layer
# - using only for orange categories so start with selection 
# 	"""Rec_Cat_short='HPC' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPD' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPM' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='MPC' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPM' and Rec_Cat in (2,3,4,5) OR
# 	Rec_Cat_short='LPC' and Rec_Cat in (2,3,4,5) OR
# 	Rec_Cat_short='LPD' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='LPM' and Rec_Cat in (2,3,4,5)"""

# - create priority p1-crosses/in CIA and TSU, p2 TSU only, p3 CIA only  
# - cumulative total for each 

sql_2a=f"""
    SELECT *,ST_AsText(geometry) as wkt, from '{ret_class}'
    WHERE
    Rec_Cat_short='HPC' and Rec_Cat in (4,5) OR
	Rec_Cat_short='HPD' and Rec_Cat in (4,5) OR
	Rec_Cat_short='HPM' and Rec_Cat in (4,5) OR
	Rec_Cat_short='MPC' and Rec_Cat in (3,4,5) OR
	Rec_Cat_short='MPD' and Rec_Cat in (3,4,5) OR
	Rec_Cat_short='MPM' and Rec_Cat in (2,3,4,5) OR
	Rec_Cat_short='LPC' and Rec_Cat in (2,3,4,5) OR
	Rec_Cat_short='LPD' and Rec_Cat in (4,5) OR
	Rec_Cat_short='LPM' and Rec_Cat in (2,3,4,5)
    """
#return query results as gdf
step_2d_base=return_gdf(sql_2a)




In [ ]:
# - remove stands that overlap/crosses with step2b 
step2b_selection=gpd.read_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))

In [ ]:
# gpd intersect difference 
step_2d_no_step2b=gpd.overlay(step_2d_base, step2b_selection, how='difference')

In [ ]:
# - remove HV1 DZ
step_2d_no_step2b=gpd.overlay(step_2d_no_step2b, hv1_dv, how='difference')

In [ ]:
base_df['geometry'] = gpd.GeoSeries.from_wkt(base_df['wkt'])
base_df = gpd.GeoDataFrame(base_df).set_crs(3005, allow_override=True)
base_df = base_df.drop(columns=['wkt'])
info(f" length of base_df: {len(base_df)}")

In [ ]:
base_df.to_parquet(os.path.join(output_dir, "step_2d_base.parquet"))

In [ ]:
cia_gdf = cia_ciap.to_crs(3005)
tsu_gdf = tsu.to_crs(3005)
cia_join = gpd.sjoin(
    base_gdf,
    cia_gdf[["geometry"]],
    how="left",
    predicate="crosses"
)

base_gdf["cia_hit"] = ~cia_join.index_right.isna()

In [ ]:
tsu_join = gpd.sjoin(
    base_gdf,
    tsu_gdf[["geometry"]],
    how="left",
    predicate="crosses"
)

base_gdf["tsu_hit"] = ~tsu_join.index_right.isna()

In [ ]:
def classify(row):
    if row.cia_hit and row.tsu_hit:
        return "cia_tsu", 1
    elif row.tsu_hit:
        return "tsu", 2
    elif row.cia_hit:
        return "cia", 3
    else:
        return None, None

base_gdf[["interactions", "hierarchy"]] = (
    base_gdf.apply(classify, axis=1, result_type="expand")
)

In [ ]:
ret_class_2d = base_gdf.copy()
print(len(ret_class_2d))

In [ ]:
ret_class_2d.to_parquet(os.path.join(output_dir, "ret_class_2d.parquet"))

In [ ]:

def _choose_prefix_for_target(sorted_df: pd.DataFrame, area_col: str, target: float):
    """
    sorted_df must already be sorted by selection priority (e.g., SIFA_2 desc, then area desc).
    Returns (index_list, picked_sum).
    """
    if sorted_df.empty or target <= 0:
        return [], 0.0

    csum = sorted_df[area_col].cumsum().to_numpy()
    idx = sorted_df.index.to_numpy()

    # position of last undershoot (<= target)
    floor_n = int((csum <= target).sum())

    if floor_n == 0:
        # first item overshoots; choose it (closest allowed)
        return [idx[0]], float(sorted_df[area_col].iloc[0])

    if floor_n == len(csum):
        # even all don't reach the target; pick all
        return list(idx), float(csum[-1])

    under_sum = float(csum[floor_n - 1])
    over_sum  = float(csum[floor_n])
    # Compare overshoot vs undershoot; tie → prefer undershoot
    if (over_sum - target) < (target - under_sum):
        pick_idx = list(idx[:floor_n + 1])
        pick_sum = over_sum
    else:
        pick_idx = list(idx[:floor_n])
        pick_sum = under_sum

    return pick_idx, pick_sum


In [ ]:

def select_by_hierarchy(
    candidates: gpd.GeoDataFrame,
    targets_long: pd.DataFrame,
    *,
    area_col: str = "aflb_area",       # or 'Area_ha' depending on your run
    age_col: str = "SIFA_2",
    rec_cat_col: str = "Rec_Cat",
    rep_col: str = "Rec_Cat_short",    # FOR_REP
    hierarchy_col: str = "hierarchy",  # 1=cia_tsu, 2=tsu, 3=cia
    # Selection priority within a Rec_Cat source pool:
    hierarchy_priority = (1, 2, 3),
    # Order in which to "move lower down" to other Rec_Cat when short (1 → 2 → 3 → 4):
    rec_cat_order = (1, 2, 3, 4),
    # Optional: how much is already met from Step 2b (so we only fill the shortfall)
    already_met: pd.DataFrame | None = None,   # columns: Rec_Cat, Rec_Cat_short, selected_area_ha
):
    """
    Returns:
      selected_gdf (GeoDataFrame): rows chosen with flags
      summary_df (DataFrame): per (Rec_Cat, Rec_Cat_short) result with deltas
      pool_remaining (GeoDataFrame): remaining (not picked) candidates
    """
    # --- Sanity & normalization ---
    gdf = candidates.copy()
    # Ensure key columns exist
    required_cols = {rec_cat_col, rep_col, area_col, age_col, hierarchy_col}
    missing = required_cols - set(gdf.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates: {missing}")

    # Normalize types/strings
    gdf[rec_cat_col] = pd.to_numeric(gdf[rec_cat_col], errors="coerce").astype("Int64")
    gdf[rep_col] = gdf[rep_col].astype(str).str.strip()
    gdf[area_col] = pd.to_numeric(gdf[area_col], errors="coerce").fillna(0.0)
    gdf[age_col] = pd.to_numeric(gdf[age_col], errors="coerce").fillna(-1e9)
    gdf[hierarchy_col] = pd.to_numeric(gdf[hierarchy_col], errors="coerce").astype("Int64")

    # Normalize targets
    tlong = targets_long.copy()
    tlong[rec_cat_col] = pd.to_numeric(tlong[rec_cat_col], errors="coerce").astype("Int64")
    tlong[rep_col] = tlong[rep_col].astype(str).str.strip()
    tlong["target_ha"] = pd.to_numeric(tlong["target_ha"], errors="coerce").fillna(0.0)

    # Already met (from 2b) → map to shortfall
    met = already_met.copy() if already_met is not None else pd.DataFrame(columns=[rec_cat_col, rep_col, "selected_area_ha"])
    if not met.empty:
        met[rec_cat_col] = pd.to_numeric(met[rec_cat_col], errors="coerce").astype("Int64")
        met[rep_col] = met[rep_col].astype(str).str.strip()
        met["selected_area_ha"] = pd.to_numeric(met["selected_area_ha"], errors="coerce").fillna(0.0)

    need = (
        tlong.merge(met[[rec_cat_col, rep_col, "selected_area_ha"]], on=[rec_cat_col, rep_col], how="left")
            .assign(selected_area_ha=lambda d: d["selected_area_ha"].fillna(0.0))
            .assign(remaining=lambda d: (d["target_ha"] - d["selected_area_ha"]).clip(lower=0.0))
    )

    # Work the targets in Rec_Cat order (1 → 2 → 3 → 4), within each by FOR_REP
    need = need.sort_values([rec_cat_col, rep_col], key=lambda s: s.map({v:i for i, v in enumerate(rec_cat_order)}).fillna(1e9))

    # Selection bookkeeping
    gdf["picked"] = False
    gdf["picked_for_rec_cat"] = pd.Series(pd.NA, index=gdf.index, dtype="Int64")
    gdf["picked_for_rep"] = pd.Series(pd.NA, index=gdf.index, dtype="object")
    gdf["picked_batch"] = pd.Series(pd.NA, index=gdf.index, dtype="Int64")  # 1,2,3 for hierarchy

    # Helper: iterate Rec_Cat sources in cascade order starting at `rc`
    def _rec_cat_cascade(rc):
        order = list(rec_cat_order)
        if rc not in order:
            return order
        start = order.index(rc)
        return order[start:]  # rc, rc+1, rc+2, ...

    # Main loop: per (Rec_Cat, Rep) shortfall
    for _, row in need.iterrows():
        target_rc   = row[rec_cat_col]
        target_rep  = row[rep_col]
        remaining   = float(row["remaining"])
        if remaining <= 0:
            continue

        # Cascade through Rec_Cat pools: start at target_rc, then lower down
        for rc_src in _rec_cat_cascade(target_rc):
            if remaining <= 0:
                break

            # Work by hierarchy priority 1 → 2 → 3
            for h in hierarchy_priority:
                if remaining <= 0:
                    break

                # Available pool: not yet picked, matching FOR_REP, source Rec_Cat, hierarchy
                pool_idx = gdf.index[
                    (~gdf["picked"]) &
                    (gdf[rep_col] == target_rep) &
                    (gdf[rec_cat_col] == rc_src) &
                    (gdf[hierarchy_col] == h)
                ]
                if len(pool_idx) == 0:
                    continue

                # Sort by age desc, then area desc
                pool = gdf.loc[pool_idx].sort_values([age_col, area_col], ascending=[False, False])

                # Choose a prefix closest to the remaining target (prefer undershoot)
                pick_idx, picked_sum = _choose_prefix_for_target(pool, area_col, remaining)
                if not pick_idx:
                    continue

                gdf.loc[pick_idx, "picked"] = True
                gdf.loc[pick_idx, "picked_for_rec_cat"] = target_rc
                gdf.loc[pick_idx, "picked_for_rep"] = target_rep
                gdf.loc[pick_idx, "picked_batch"] = h

                remaining -= picked_sum

    # Build outputs
    selected_gdf = gdf[gdf["picked"]].copy()
    pool_remaining = gdf[~gdf["picked"]].copy()

    # Summary against the *requested* target (not the “already_met”)
    # Join back target_ha for the (rec_cat, rep) the feature was picked for
    picked_key = selected_gdf[[rec_cat_col, rep_col]].rename(
        columns={rec_cat_col: "picked_rc", rep_col: "picked_rep"}
    )
    # Aggregate by what they were picked for:
    summary = (
        selected_gdf
        .groupby(["picked_for_rec_cat", "picked_for_rep"], as_index=False)
        .agg(selected_area_ha=(area_col, "sum"),
             n_features=(area_col, "size"),
             max_SIFA2=(age_col, "max"),
             min_SIFA2=(age_col, "min"))
        .rename(columns={"picked_for_rec_cat": rec_cat_col, "picked_for_rep": rep_col})
        .merge(tlong[[rec_cat_col, rep_col, "target_ha"]], on=[rec_cat_col, rep_col], how="left")
    )
    summary["delta_to_target"] = summary["selected_area_ha"] - summary["target_ha"]
    summary["pct_met"] = np.where(summary["target_ha"] > 0,
                                  (summary["selected_area_ha"] / summary["target_ha"]) * 100.0,
                                  np.nan)

    return selected_gdf, summary, pool_remaining


In [ ]:

# 1) candidates: the output after step 2c (no HV1) + step 2d anti-join (not in step2b)
#    Must contain: Rec_Cat, Rec_Cat_short, SIFA_2, aflb_area (or Area_ha), hierarchy (1/2/3)
candidates = ret_class_2c  # GeoDataFrame from your DuckDB query

# 2) targets_long: you already constructed it from CSV
#    columns: Rec_Cat (Int64), Rec_Cat_short (str), target_ha (float)

# 3) already_met: take from your Step 2b summary if available
#    columns: Rec_Cat, Rec_Cat_short, selected_area_ha
already_met = summary_2b[['Rec_Cat','Rec_Cat_short','selected_area_ha']].copy()  # example

# 4) Run the selector
selected_gdf, summary_2d, pool_remaining = select_by_hierarchy(
    candidates=candidates,
    targets_long=targets_long,
    area_col="aflb_area",          # or "Area_ha"
    age_col="SIFA_2",
    rec_cat_col="Rec_Cat",
    rep_col="Rec_Cat_short",
    hierarchy_col="hierarchy",
    hierarchy_priority=(1,2,3),
    rec_cat_order=(1,2,3,4),
    already_met=already_met        # or None if you want to hit full targets
)

# 5) Save results
selected_gdf.to_parquet(os.path.join(output_dir, "step_2d_selection.parquet"), index=False)
summary_2d.to_csv(os.path.join(output_dir, "step_2d_summary.csv"), index=False)
